# Riboflavin: VIMCO-$\star$ training

Collapsed spike-and-slab regression with $\tau^2=1$. Run from this folder. Fits every tuning configuration for every train-test split (seeds 1000-1049) and writes PIPs, logits and run metadata to `the output folder`.

In [ ]:
%load_ext Cython


In [ ]:
%%cython --compile-args=-O3
# cython: language_level=3
# cython: boundscheck=False
# cython: wraparound=False
# cython: cdivision=True
# cython: initializedcheck=False

import os
import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

device = torch.device('cpu')
dtype = torch.float64

SPLIT_SEEDS = range(1000, 1050)
METHOD_SEED = 12345
METHOD_NAME = 'VIMCO*'

tau2 = 1.0
prior_prec = 1.0 / tau2
jitter = 1e-8
PRIOR_EXPECTED_MODEL_SIZE = 6.0
b_pi = 1.0
a_pi = None

TOTAL_SAMPLES = 300
S_COPIES_THIS_FILE = 1
N_IMPORTANCE_THIS_FILE = 300
split_label = f'S_{S_COPIES_THIS_FILE}_N_{N_IMPORTANCE_THIS_FILE}'

LR_GRID = [0.01, 0.05, 0.1, 0.2, 0.3]
ALPHA_GRID = [0.2, 0.5, 0.7, 0.9]

PATIENCE_MAX = 200
MAX_ITER = 3000
BOUND_TOL = 1e-4
MAX_GRAD_NORM = 5.0

ADAPTIVE_ACTIVE_MAX_Q = 50
ACTIVE_CHOL_BATCH_SIZE = 64
DUAL_FALLBACK_BATCH_SIZE = 8

ROOT = Path.cwd().resolve()
output_base_root = ROOT / 'output_riboflavin_prediction_train50_PIP_lambda_only_tau2_1_final'
output_root = output_base_root / 'm0_6'
output_root.mkdir(parents=True, exist_ok=True)

NUM_TORCH_THREADS = 2
torch.set_num_threads(NUM_TORCH_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

SPLIT_SEEDS_LIST = list(SPLIT_SEEDS)

SPLIT_ROOT_CANDIDATES = [
    ROOT / 'riboflavin_prediction_splits_50x_50train_21test',
    ROOT / 'ribo' / 'riboflavin_prediction_splits_50x_50train_21test',
    ROOT.parent / 'ribo' / 'riboflavin_prediction_splits_50x_50train_21test',
]

split_data_root = next(
    (path for path in SPLIT_ROOT_CANDIDATES if path.exists()),
    SPLIT_ROOT_CANDIDATES[0],
)
if not split_data_root.exists():
    raise FileNotFoundError(
        'Cannot find riboflavin_prediction_splits_50x_50train_21test. Tried:\n'
        + '\n'.join(str(path) for path in SPLIT_ROOT_CANDIDATES)
    )

SPLIT_SEED_TO_REP = {
    int(split_seed): int(split_seed - 1000)
    for split_seed in SPLIT_SEEDS_LIST
}

def train_file_for_split_seed(split_seed):
    split_seed = int(split_seed)
    rep = SPLIT_SEED_TO_REP[split_seed]
    path = split_data_root / f'rep_{rep:02d}_seed_{split_seed}' / 'train.csv'
    if not path.exists():
        raise FileNotFoundError(f'Cannot find training file: {path}')
    return path

_first_train_file = train_file_for_split_seed(SPLIT_SEEDS_LIST[0])
_header_df = pd.read_csv(_first_train_file, nrows=1)
if 'y' not in _header_df.columns:
    raise ValueError("Each train.csv must contain a response column named 'y'.")

feature_cols = [
    col for col in _header_df.columns
    if col != 'y' and not str(col).startswith('Unnamed')
]
p = int(len(feature_cols))

if not (0.0 < PRIOR_EXPECTED_MODEL_SIZE < p):
    raise ValueError('PRIOR_EXPECTED_MODEL_SIZE must be between 0 and p.')

a_pi = b_pi * PRIOR_EXPECTED_MODEL_SIZE / (p - PRIOR_EXPECTED_MODEL_SIZE)

print('split_data_root =', split_data_root)
print('method =', METHOD_NAME, '|', split_label)
print('p =', p, '| n_train = 50')
print('X is already standardised in train.csv and is NOT standardised again.')
print('y is left uncentred / on its original scale.')
print('a_pi =', a_pi, '| b_pi =', b_pi)
print('output_root =', output_root)

log_2pi_t = float(np.log(2.0 * np.pi))
tau2_t = torch.tensor(tau2, dtype=dtype, device=device)
prior_prec_t = torch.tensor(prior_prec, dtype=dtype, device=device)
log_prior_prec_t = torch.log(prior_prec_t)

a_pi_t = torch.tensor(a_pi, dtype=dtype, device=device)
b_pi_t = torch.tensor(b_pi, dtype=dtype, device=device)
p_ab_t = torch.tensor(float(p + a_pi + b_pi), dtype=dtype, device=device)
LOG_BETA_AB_t = torch.lgamma(a_pi_t) + torch.lgamma(b_pi_t) - torch.lgamma(a_pi_t + b_pi_t)
_MODEL_SIZE_GRID_t = torch.arange(p + 1, dtype=dtype, device=device)
_LOG_MODEL_PRIOR_BY_SIZE_t = (
    torch.lgamma(a_pi_t + _MODEL_SIZE_GRID_t)
    + torch.lgamma(b_pi_t + p - _MODEL_SIZE_GRID_t)
    - torch.lgamma(p_ab_t)
    - LOG_BETA_AB_t
)

n = None
nu_t = None
marginal_constant_t = None
XtX_t = None
Xty_t = None
yty_t = None
Xc_T_t = None
yc_t = None
ycyc_t = None
log_n_t = None
I_n_t = None
_REGULARISATION_CACHE = {}

def regularisation_matrix_cached(q):
    matrix = _REGULARISATION_CACHE.get(q)
    if matrix is None:
        prior_diag = torch.full((q,), prior_prec, dtype=dtype, device=device)
        prior_diag[0] = 0.0
        matrix = torch.diag(prior_diag) + jitter * torch.eye(q, dtype=dtype, device=device)
        _REGULARISATION_CACHE[q] = matrix
    return matrix

def load_dataset(split_seed):
    global n, nu_t, marginal_constant_t
    global XtX_t, Xty_t, yty_t
    global Xc_T_t, yc_t, ycyc_t, log_n_t, I_n_t

    dataset_file = train_file_for_split_seed(split_seed)
    data_df = pd.read_csv(dataset_file)

    X_np = np.ascontiguousarray(
        data_df[feature_cols].to_numpy(dtype=np.float64), dtype=np.float64
    )
    y_np = np.ascontiguousarray(
        data_df['y'].to_numpy(dtype=np.float64), dtype=np.float64
    )

    n = int(X_np.shape[0])
    if n != 50 or X_np.shape != (50, p):
        raise ValueError(
            f'Expected train shape X=(50,{p}), y=(50,), got X={X_np.shape}, y={y_np.shape}.'
        )
    if not np.all(np.isfinite(X_np)) or not np.all(np.isfinite(y_np)):
        raise ValueError('Training split contains non-finite values.')

    max_abs_mean = float(np.max(np.abs(X_np.mean(axis=0))))
    max_abs_sd_error = float(np.max(np.abs(X_np.std(axis=0, ddof=0) - 1.0)))
    if max_abs_mean > 1e-6 or max_abs_sd_error > 1e-6:
        print(
            f'WARNING split_seed={split_seed}: X does not look exactly standardised '
            f'(max|mean|={max_abs_mean:.3e}, max|sd-1|={max_abs_sd_error:.3e}). '
            'No re-standardisation was applied.'
        )

    nu_t = torch.tensor(float(n - 1), dtype=dtype, device=device)
    marginal_constant_t = torch.lgamma(0.5 * nu_t) - 0.5 * nu_t * log_2pi_t

    X_full_np = np.column_stack([np.ones(n, dtype=np.float64), X_np])
    XtX_t = torch.from_numpy(np.asarray(X_full_np.T @ X_full_np, dtype=np.float64)).to(
        device=device, dtype=dtype
    )
    Xty_t = torch.from_numpy(np.asarray(X_full_np.T @ y_np, dtype=np.float64)).to(
        device=device, dtype=dtype
    )
    yty_t = torch.tensor(float(y_np @ y_np), dtype=dtype, device=device)

    Xc_np = np.ascontiguousarray(X_np - X_np.mean(axis=0, keepdims=True), dtype=np.float64)
    yc_np = np.ascontiguousarray(y_np - y_np.mean(), dtype=np.float64)
    Xc_T_t = torch.from_numpy(Xc_np).to(device=device, dtype=dtype).t().contiguous()
    yc_t = torch.from_numpy(yc_np).to(device=device, dtype=dtype)
    ycyc_t = torch.sum(yc_t.square())
    log_n_t = torch.log(torch.tensor(float(n), dtype=dtype, device=device))
    I_n_t = torch.eye(n, dtype=dtype, device=device)

    _REGULARISATION_CACHE.clear()
    return dataset_file

def gamma_from_eta(eta):
    return torch.sigmoid(eta)

def sample_G(gamma, total_samples):
    return torch.bernoulli(gamma.expand(total_samples, -1))

def log_q_eta(G, eta):
    return torch.sum(G * F.logsigmoid(eta) + (1.0 - G) * F.logsigmoid(-eta), dim=1)

def make_initial_eta():

    prior_mean = a_pi / (a_pi + b_pi)
    initial_lambda = np.log(prior_mean) - np.log1p(-prior_mean)
    return torch.full((p,), float(initial_lambda), dtype=dtype, device=device)

def log_model_prior_torch(G):
    model_size = G.sum(dim=1).to(torch.long)
    return _LOG_MODEL_PRIOR_BY_SIZE_t[model_size]

def log_marginal_likelihood_active_torch(G):
    number_models = int(G.shape[0])
    out = torch.empty(number_models, dtype=dtype, device=device)
    active_counts = G.sum(dim=1).to(torch.long)
    unique_counts = torch.unique(active_counts)

    for count_tensor in unique_counts:
        k_slope = int(count_tensor.item())
        row_indices_all = torch.nonzero(active_counts == k_slope, as_tuple=False).flatten()
        q = k_slope + 1

        for start in range(0, int(row_indices_all.numel()), ACTIVE_CHOL_BATCH_SIZE):
            row_indices = row_indices_all[start:start + ACTIVE_CHOL_BATCH_SIZE]
            batch_size = int(row_indices.numel())

            if k_slope == 0:
                idx_mat = torch.zeros((batch_size, 1), dtype=torch.long, device=device)
            else:
                group_G = G.index_select(0, row_indices)
                nonzero_locations = torch.nonzero(group_G > 0.5, as_tuple=False)
                slope_idx = nonzero_locations[:, 1].reshape(batch_size, k_slope).to(torch.long) + 1
                intercept = torch.zeros((batch_size, 1), dtype=torch.long, device=device)
                idx_mat = torch.cat((intercept, slope_idx), dim=1)

            XtX_batch = XtX_t[idx_mat[:, :, None], idx_mat[:, None, :]]
            Xty_batch = Xty_t[idx_mat]
            M_batch = XtX_batch + regularisation_matrix_cached(q).unsqueeze(0)
            L_batch = torch.linalg.cholesky(M_batch)
            solve_batch = torch.cholesky_solve(Xty_batch[..., None], L_batch).squeeze(-1)
            residual_batch = yty_t - torch.sum(Xty_batch * solve_batch, dim=1)
            residual_batch = torch.clamp(residual_batch, min=jitter)
            logdet_M_batch = 2.0 * torch.sum(
                torch.log(torch.diagonal(L_batch, dim1=-2, dim2=-1)), dim=1
            )
            out[row_indices] = (
                marginal_constant_t
                + 0.5 * k_slope * log_prior_prec_t
                - 0.5 * logdet_M_batch
                - 0.5 * nu_t * torch.log(residual_batch / 2.0)
            )

    return out

def log_marginal_likelihood_dual_internal_torch(G):
    number_models = int(G.shape[0])
    out = torch.empty(number_models, dtype=dtype, device=device)
    active_counts = G.sum(dim=1).to(torch.long)
    unique_counts = torch.unique(active_counts)

    for count_tensor in unique_counts:
        k_slope = int(count_tensor.item())
        row_indices = torch.nonzero(active_counts == k_slope, as_tuple=False).flatten()
        group_size = int(row_indices.numel())

        if k_slope == 0:
            residual = torch.clamp(ycyc_t, min=jitter)
            value = marginal_constant_t - 0.5 * log_n_t - 0.5 * nu_t * torch.log(residual / 2.0)
            out[row_indices] = value.expand(group_size)
            continue

        group_G = G.index_select(0, row_indices)
        nonzero_locations = torch.nonzero(group_G > 0.5, as_tuple=False)
        slope_idx = nonzero_locations[:, 1].reshape(group_size, k_slope).to(torch.long)

        for start in range(0, group_size, DUAL_FALLBACK_BATCH_SIZE):
            end = min(start + DUAL_FALLBACK_BATCH_SIZE, group_size)
            current_rows = row_indices[start:end]
            current_idx = slope_idx[start:end]
            current_batch_size = int(current_idx.shape[0])

            XA = Xc_T_t[current_idx].transpose(1, 2).contiguous()
            K_batch = I_n_t.unsqueeze(0) + tau2_t * torch.bmm(XA, XA.transpose(1, 2))
            if jitter > 0:
                K_batch = K_batch + jitter * I_n_t.unsqueeze(0)

            L_batch = torch.linalg.cholesky(K_batch)
            y_batch = yc_t.expand(current_batch_size, -1).unsqueeze(2)
            solved = torch.cholesky_solve(y_batch, L_batch).squeeze(2)
            residual_batch = torch.sum(yc_t.unsqueeze(0) * solved, dim=1)
            residual_batch = torch.clamp(residual_batch, min=jitter)
            logdet_K_batch = 2.0 * torch.sum(
                torch.log(torch.diagonal(L_batch, dim1=-2, dim2=-1)), dim=1
            )
            out[current_rows] = (
                marginal_constant_t
                - 0.5 * log_n_t
                - 0.5 * logdet_K_batch
                - 0.5 * nu_t * torch.log(residual_batch / 2.0)
            )

    return out

def log_marginal_likelihood_adaptive_torch(G):
    number_models = int(G.shape[0])
    out = torch.empty(number_models, dtype=dtype, device=device)
    active_counts = G.sum(dim=1).to(torch.long)
    active_route_mask = (active_counts + 1) <= int(ADAPTIVE_ACTIVE_MAX_Q)

    if bool(active_route_mask.any().item()):
        idx = torch.nonzero(active_route_mask, as_tuple=False).flatten()
        out[idx] = log_marginal_likelihood_active_torch(G.index_select(0, idx))

    if bool((~active_route_mask).any().item()):
        idx = torch.nonzero(~active_route_mask, as_tuple=False).flatten()
        out[idx] = log_marginal_likelihood_dual_internal_torch(G.index_select(0, idx))

    return out

def log_joint_integrated_torch(G):
    return log_marginal_likelihood_adaptive_torch(G) + log_model_prior_torch(G)

TRAINING_CODE_SHA256 = "088c2dee1d8f0bf8c47f172181cee4b78c0718b369b473fac857068ad23db00e"

def clean_float(x):
    return str(x).replace('.', 'p').replace('-', 'm')


def safe_method_name(method):
    return method.replace('*', 'star').replace('-', '_').replace(' ', '_').lower()


def method_output_dir():
    folder = output_root / f"output_{safe_method_name(METHOD_NAME)}_riboflavin"
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = Path(str(path) + '.tmp')
    frame.to_csv(temporary, index=False, float_format='%.17g')
    os.replace(temporary, path)


def settings():
    for alpha in ALPHA_GRID:
        for lr in LR_GRID:
            yield lr, alpha, 1, 300, f'star_alpha_{clean_float(alpha)}_lr_{clean_float(lr)}_S_1_N_300'


def fit_config(split_seed, train_hash, learning_rate, alpha, S, N):
    return {
        'method': METHOD_NAME, 'split_seed': int(split_seed), 'METHOD_SEED': int(METHOD_SEED),
        'tau2': float(tau2), 'a_pi': float(a_pi), 'b_pi': float(b_pi),
        'learning_rate': float(learning_rate), 'alpha_setting': alpha, 'S': int(S), 'N': int(N),
        'LR_GRID': LR_GRID, 'ALPHA_GRID': globals().get('ALPHA_GRID'),
        'PATIENCE_MAX': PATIENCE_MAX, 'MAX_ITER': MAX_ITER,
        'BOUND_TOL': BOUND_TOL, 'MAX_GRAD_NORM': MAX_GRAD_NORM,
        'ADAPTIVE_ACTIVE_MAX_Q': ADAPTIVE_ACTIVE_MAX_Q,
        'ACTIVE_CHOL_BATCH_SIZE': ACTIVE_CHOL_BATCH_SIZE,
        'DUAL_FALLBACK_BATCH_SIZE': DUAL_FALLBACK_BATCH_SIZE,
        'jitter': jitter, 'expected_model_size': PRIOR_EXPECTED_MODEL_SIZE,
        'annealing': ({'start': globals().get('ALPHA_START'), 'minimum': globals().get('ALPHA_MIN'),
                       'step': globals().get('ALPHA_STEP'), 'ess_threshold': globals().get('ESS_LOW')}
                      if 'ALPHA_START' in globals() else None),
        'train_file_sha256': train_hash, 'training_code_sha256': TRAINING_CODE_SHA256,
    }


def estimate_star_f_scaled_from_batch(G, logw, gamma, alpha):
    eps = 1e-12
    one_minus_alpha = max(1e-12, 1.0 - float(alpha))
    logu_abs = one_minus_alpha * logw
    shift = torch.max(logu_abs)
    u_scaled = torch.exp(logu_abs - shift)
    score = G - gamma[None, :]
    a10 = u_scaled.mean()
    a11 = (u_scaled[:, None] * score).mean(dim=0)
    a12 = (u_scaled[:, None] * score.square()).mean(dim=0)
    a01 = score.mean(dim=0)
    a02 = score.square().mean(dim=0)
    num = a12 - (a11 * a11) / (a10 + eps)
    den = (a02 - a01 * a01) + eps
    f_scaled = torch.clamp(alpha * (num / den), min=0.0)
    f_scaled = torch.nan_to_num(f_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    return f_scaled.detach(), float(shift.item())


def estimate_star_f_loo_scaled(G, u_scaled, gamma, alpha):
    eps = 1e-12
    N = int(G.shape[0])
    if N <= 1:
        raise ValueError("Exact leave-one-out VIMCO* requires N > 1.")
    if alpha <= 0.0:
        return torch.zeros_like(G)
    score = G - gamma[None, :]
    score2 = score.square()
    u_sum = torch.sum(u_scaled)
    score_sum = torch.sum(score, dim=0)
    score2_sum = torch.sum(score2, dim=0)
    u_score_sum = torch.sum(u_scaled[:, None] * score, dim=0)
    u_score2_sum = torch.sum(u_scaled[:, None] * score2, dim=0)
    n_loo = float(N - 1)
    a10 = (u_sum - u_scaled) / n_loo
    a11 = (u_score_sum[None, :] - u_scaled[:, None] * score) / n_loo
    a12 = (u_score2_sum[None, :] - u_scaled[:, None] * score2) / n_loo
    a01 = (score_sum[None, :] - score) / n_loo
    a02 = (score2_sum[None, :] - score2) / n_loo
    num = a12 - (a11 * a11) / (a10[:, None] + eps)
    den = (a02 - a01 * a01) + eps
    f_loo_scaled = torch.clamp(alpha * (num / den), min=0.0)
    f_loo_scaled = torch.nan_to_num(f_loo_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    return f_loo_scaled.detach()


def lagged_log_inside(wbar, sum_u_scaled, current_shift, f_prev_scaled, f_prev_shift):
    tiny = 1e-300
    base = torch.clamp(1.0 - wbar, min=tiny)
    log_base = torch.log(base)[:, None]
    positive = f_prev_scaled > 0.0
    neg_inf = torch.full_like(f_prev_scaled, -float("inf"))
    log_f_scaled = torch.where(positive, torch.log(f_prev_scaled), neg_inf)
    log_ratio = log_f_scaled + float(f_prev_shift) - current_shift - torch.log(sum_u_scaled)
    return torch.logaddexp(log_base, log_ratio[None, :])


def vimco_star_grad_eta(eta, total_samples, S_copies, alpha, f_prev_scaled, f_prev_shift):
    one_minus_alpha = max(1e-12, 1.0 - float(alpha))
    N_importance = total_samples // S_copies
    gamma = gamma_from_eta(eta)
    G_total = sample_G(gamma, total_samples)
    logp_total = log_joint_integrated_torch(G_total)
    logq_total = log_q_eta(G_total, eta)
    logw_total = logp_total - logq_total
    if not bool(torch.isfinite(logw_total).all().item()):
        raise FloatingPointError("Non-finite VIMCO* log importance weight detected.")
    f_new_scaled, f_new_shift = estimate_star_f_scaled_from_batch(G_total, logw_total, gamma, alpha)
    G_blocks = G_total.reshape(S_copies, N_importance, p)
    logw_blocks = logw_total.reshape(S_copies, N_importance)
    grad_sum = torch.zeros(p, dtype=dtype, device=device)
    bound_sum = torch.tensor(0.0, dtype=dtype, device=device)

    for r in range(S_copies):
        G = G_blocks[r]
        logw = logw_blocks[r]
        logu_abs = one_minus_alpha * logw
        current_shift = torch.max(logu_abs)
        u_scaled = torch.exp(logu_abs - current_shift)
        sum_u_scaled = torch.sum(u_scaled)
        wbar = u_scaled / sum_u_scaled
        score = G - gamma[None, :]
        term1 = -torch.sum(wbar[:, None] * score, dim=0)
        if f_prev_scaled is None:
            f_loo_scaled = estimate_star_f_loo_scaled(G, u_scaled, gamma, alpha)
            inside = 1.0 - wbar[:, None] + f_loo_scaled / sum_u_scaled
            log_inside = torch.log(torch.clamp(inside, min=1e-300))
        else:
            log_inside = lagged_log_inside(wbar, sum_u_scaled, current_shift, f_prev_scaled, f_prev_shift)
        term2 = -torch.sum(score * log_inside, dim=0) / one_minus_alpha
        grad_sum += term1 + term2
        bound_r = (current_shift + torch.log(sum_u_scaled) - np.log(N_importance)) / one_minus_alpha
        bound_sum += bound_r

    grad = grad_sum / S_copies
    bound = bound_sum / S_copies
    if not bool(torch.isfinite(grad).all().item()):
        raise FloatingPointError("Non-finite VIMCO* gradient detected.")
    return grad, f_new_scaled.detach(), float(f_new_shift), float(bound.item())


def run_vimco_star(lr, S_copies, N_importance, method_seed, alpha):
    torch.manual_seed(int(method_seed))
    np.random.seed(int(method_seed))

    total_samples = int(S_copies) * int(N_importance)
    eta = torch.nn.Parameter(make_initial_eta())
    optimizer = torch.optim.Adam([eta], lr=lr, betas=(0.9, 0.999))
    current_alpha = float(alpha)
    f_prev_scaled = None
    f_prev_shift = None
    best_training_bound = -float('inf')
    best_eta = None
    patience = 0

    for it in range(1, MAX_ITER + 1):
        optimizer.zero_grad(set_to_none=True)
        alpha_used = float(current_alpha)

        with torch.no_grad():
            grad, f_new_scaled, f_new_shift, training_bound = vimco_star_grad_eta(
                eta, total_samples, S_copies, alpha_used, f_prev_scaled, f_prev_shift
            )

        if training_bound > best_training_bound + BOUND_TOL:
            best_training_bound = training_bound
            best_eta = eta.detach().clone()
            patience = 0
        else:
            if training_bound > best_training_bound:
                best_training_bound = training_bound
                best_eta = eta.detach().clone()
            patience += 1

        eta.grad = (-grad).clone()
        torch.nn.utils.clip_grad_norm_([eta], max_norm=MAX_GRAD_NORM)
        optimizer.step()
        f_prev_scaled = f_new_scaled.clone()
        f_prev_shift = float(f_new_shift)

        if patience >= PATIENCE_MAX:
            break

    if best_eta is not None:
        with torch.no_grad():
            eta.copy_(best_eta)

    return {
        'pip': gamma_from_eta(eta).detach().cpu().numpy(),
        'lambda': eta.detach().cpu().numpy(),
        'iterations': int(it),
        'final_alpha': float(alpha),
    }


def train_and_save_one_split(split_seed):
    train_path = load_dataset(split_seed)
    train_hash = hashlib.sha256(Path(train_path).read_bytes()).hexdigest()
    output_dir = method_output_dir()
    stem = f'{safe_method_name(METHOD_NAME)}_{split_label}_splitseed_{split_seed}_methodseed_{METHOD_SEED}.csv'
    pip_file = output_dir / f'pips_lambda_{stem}'
    summary_file = output_dir / f'training_summary_{stem}'
    pips = {'variable_name': feature_cols}
    rows = []
    print(f'\n{METHOD_NAME} | split_seed={split_seed} | tau2={tau2:g} | S,N={S_COPIES_THIS_FILE},{N_IMPORTANCE_THIS_FILE}', flush=True)
    for lr, alpha, S, N, label in settings():
        config = fit_config(split_seed, train_hash, lr, alpha, S, N)
        signature = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
        print(f'  train {label}', flush=True)
        started = time.perf_counter()
        result = run_vimco_star(lr, S, N, METHOD_SEED, alpha)
        runtime = time.perf_counter() - started
        pip = np.asarray(result['pip'], dtype=np.float64)
        eta = np.asarray(result['lambda'], dtype=np.float64)
        if (pip.shape != (p,) or eta.shape != (p,) or not np.isfinite(pip).all()
                or not np.isfinite(eta).all()
                or not np.allclose(pip, 1.0 / (1.0 + np.exp(-eta)), atol=1e-12, rtol=0)):
            raise ValueError(f'Invalid PIPs/logits: split {split_seed}, {label}')
        pip_col, lambda_col = f'pip_{label}', f'lambda_{label}'
        pips[pip_col], pips[lambda_col] = pip, eta
        rows.append({
            'method': METHOD_NAME, 'split_label': split_label, 'setting': label,
            'split_seed': int(split_seed), 'method_seed': METHOD_SEED,
            'learning_rate': float(lr), 'alpha_setting': alpha,
            'S': int(S), 'N': int(N), 'tau2': float(tau2),
            'a_pi': float(a_pi), 'b_pi': float(b_pi),
            'runtime_seconds': float(runtime), 'iterations': int(result['iterations']),
            'final_alpha': result['final_alpha'], 'saved_alpha': result.get('saved_alpha', np.nan),
            'best_iter': result.get('best_iter', np.nan),
            'training_bound_saved': result.get('best_training_bound', np.nan),
            'stop': result.get('stop', 'patience' if result['iterations'] < MAX_ITER else 'max_iter'),
            'pip_file': str(pip_file), 'pip_column': pip_col, 'lambda_column': lambda_col,
            'config_sha256': signature, 'train_file_sha256': train_hash,
            'training_code_sha256': TRAINING_CODE_SHA256,
        })
    atomic_csv(pd.DataFrame(pips), pip_file)
    atomic_csv(pd.DataFrame(rows), summary_file)
    return pip_file

for split_seed in SPLIT_SEEDS_LIST:
    train_and_save_one_split(split_seed)
print('Done. PIPs, logits and training metadata:', output_root)
